# End-to-End Medical RAG System
## Supabase + Auto-detect embedding (OpenAI or Ollama) + Ollama/OpenAI Chat

**Embedding fallback logic (768 dims for both):**
- If `OPENAI_API_KEY` is set → `text-embedding-3-small` with `dimensions=768`
- Otherwise → Ollama `nomic-embed-text` (768d) or set `OLLAMA_EMBED_MODEL=ncats/MedCPT-Article-Encoder` after manual GGUF import

Both produce 768-dim vectors, so the same Supabase table works regardless of which provider you use.

In [ ]:
# INSTALL (run once)
!pip install pypdf langchain langchain-text-splitters
!pip install supabase openai pandas tqdm numpy ollama requests

In [ ]:
from supabase import create_client
from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from openai import OpenAI
from tqdm import tqdm
import requests
import json
import os
import ollama

In [ ]:
# ── Credentials (set as env vars or replace defaults) ────────────────────────
SUPABASE_URL         = os.getenv("SUPABASE_URL", "https://jyhxlbxqwrbkhbfovbdf.supabase.co")
SUPABASE_SERVICE_KEY = os.getenv("SUPABASE_SERVICE_KEY", "YOUR_SUPABASE_SERVICE_ROLE_KEY")
OPENAI_API_KEY       = os.getenv("OPENAI_API_KEY", "")  # leave blank to use Ollama

# ── Embedding config ─────────────────────────────────────────────────────────
# Both providers are normalised to 768 dims — DO NOT change this
EMBED_DIMS = 768

# Ollama settings (used when OPENAI_API_KEY is blank)
OLLAMA_BASE_URL  = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
# Options: nomic-embed-text (easy), MedCPT needs manual GGUF import
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "nomic-embed-text")

# ── Chat config ───────────────────────────────────────────────────────────────
TOP_K        = 5
USE_OLLAMA   = True          # True = Ollama chat; False = OpenAI chat
OLLAMA_MODEL = "llama3.1:8b"
OPENAI_MODEL = "gpt-4o-mini"

In [ ]:
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

USE_OPENAI_EMBED = bool(OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"))
embed_provider = f"OpenAI text-embedding-3-small ({EMBED_DIMS}d)" if USE_OPENAI_EMBED else f"Ollama {OLLAMA_EMBED_MODEL} ({EMBED_DIMS}d)"
print(f"Embedding provider : {embed_provider}")
print(f"Chat provider      : {'Ollama ' + OLLAMA_MODEL if USE_OLLAMA else 'OpenAI ' + OPENAI_MODEL}")

In [ ]:
def read_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

def create_chunks(text):
    return splitter.split_text(text)

In [ ]:
# ── Embedding helpers ─────────────────────────────────────────────────────────

def _embed_openai(texts):
    """Batch-embed via OpenAI with dimensions=768 (native truncation)."""
    BATCH = 100
    all_embeddings = []
    for i in tqdm(range(0, len(texts), BATCH), desc="OpenAI embedding batches"):
        batch = texts[i:i + BATCH]
        resp = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=batch,
            dimensions=EMBED_DIMS
        )
        sorted_data = sorted(resp.data, key=lambda x: x.index)
        all_embeddings.extend([d.embedding for d in sorted_data])
    return all_embeddings


def _embed_ollama(texts):
    """Embed via Ollama /api/embeddings (sequential — no batch endpoint)."""
    all_embeddings = []
    for i, text in enumerate(tqdm(texts, desc=f"Ollama {OLLAMA_EMBED_MODEL} embedding")):
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/embeddings",
            json={"model": OLLAMA_EMBED_MODEL, "prompt": text}
        )
        resp.raise_for_status()
        data = resp.json()
        if "embedding" not in data:
            raise ValueError(f"Ollama returned no embedding for model '{OLLAMA_EMBED_MODEL}'. Is it pulled?")
        all_embeddings.append(data["embedding"])
    return all_embeddings


def generate_embedding(text):
    """Embed a single text using the active provider."""
    return batch_embed([text])[0]


def batch_embed(texts):
    """Embed a list of texts. Auto-selects OpenAI or Ollama based on OPENAI_API_KEY."""
    if not texts:
        return []
    if USE_OPENAI_EMBED:
        return _embed_openai(texts)
    return _embed_ollama(texts)

In [ ]:
def store_chunk(title, chunk_id, content, embedding):
    payload = {
        "source": "research_paper",
        "title": title,
        "chunk_id": chunk_id,
        "content": content,
        "embedding": embedding,
        "metadata": {"domain": "medical", "embed_model": OLLAMA_EMBED_MODEL if not USE_OPENAI_EMBED else "text-embedding-3-small"}
    }
    return supabase.table("medical_documents").upsert(
        payload, on_conflict="title,chunk_id"
    ).execute()

In [ ]:
def ingest_pdf(pdf_path):
    """Ingest a PDF: extract → chunk → batch-embed → upsert. Skips already-ingested chunks."""
    title = os.path.basename(pdf_path)
    text  = read_pdf(pdf_path)
    if not text.strip():
        print("WARNING: no text extracted from", pdf_path)
        return

    chunks = create_chunks(text)
    print(f"Total chunks: {len(chunks)}")

    # Skip already-ingested chunks (duplicate prevention)
    existing = supabase.table("medical_documents") \
        .select("chunk_id") \
        .eq("title", title) \
        .execute()
    existing_ids = {row["chunk_id"] for row in (existing.data or [])}
    new_chunks = [(idx, c) for idx, c in enumerate(chunks) if idx not in existing_ids]
    print(f"New chunks to embed: {len(new_chunks)} (skipping {len(existing_ids)} existing)")

    if not new_chunks:
        print("Nothing to ingest — all chunks already in Supabase.")
        return

    print(f"Embedding with: {embed_provider}")
    embeddings = batch_embed([c for _, c in new_chunks])

    print("Storing to Supabase…")
    for (idx, content), emb in tqdm(zip(new_chunks, embeddings), total=len(new_chunks)):
        store_chunk(title, idx, content, emb)

    print(f"Done: '{title}' — {len(new_chunks)} new chunks ingested.")

In [ ]:
def retrieve(query, top_k=TOP_K):
    query_embedding = generate_embedding(query)
    response = supabase.rpc(
        "match_medical_documents",
        {"query_embedding": query_embedding, "match_count": top_k}
    ).execute()
    return response.data

In [ ]:
def build_context(results):
    return "\n\n".join([
        f"[{i+1}] {r['title']}\n{r['content']}"
        for i, r in enumerate(results)
    ])

In [ ]:
def confidence_label(score):
    if score > 0.85:  return "High"
    if score > 0.75:  return "Medium"
    return "Low"

In [ ]:
def generate_answer(query, context):
    prompt = f"""You are a medical AI assistant.

Use ONLY the supplied medical context.

Context:
{context}

Question:
{query}

Rules:
- Be medically accurate
- Use simple language
- Include educational disclaimer
"""
    if USE_OLLAMA:
        try:
            response = ollama.chat(
                model=OLLAMA_MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
            return response["message"]["content"]
        except Exception as e:
            print("Ollama chat failed, fallback to OpenAI:", e)

    if not openai_client:
        raise RuntimeError("No OpenAI key set and Ollama unavailable.")
    result = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return result.choices[0].message.content

In [ ]:
def save_chat_history(session_id, query, response, confidence, similarity):
    payload = {
        "session_id": session_id,
        "query": query,
        "response": response,
        "confidence": confidence,
        "similarity": similarity,
        "llm_used": "ollama" if USE_OLLAMA else "openai"
    }
    supabase.table("chat_history").insert(payload).execute()

In [ ]:
def medical_rag(query, session_id="demo"):
    results = retrieve(query)
    if not results:
        return {"answer": "No relevant documents found."}

    context    = build_context(results)
    answer     = generate_answer(query, context)
    similarity = results[0]["similarity"]
    confidence = confidence_label(similarity)

    save_chat_history(session_id, query, answer, confidence, similarity)

    return {
        "query":      query,
        "answer":     answer,
        "confidence": confidence,
        "similarity": similarity,
        "sources":    list(dict.fromkeys(r["title"] for r in results)),
        "embed_via":  embed_provider,
        "llm_used":   "ollama" if USE_OLLAMA else "openai"
    }

In [ ]:
# ── Usage ─────────────────────────────────────────────────────────────────────
# Step 1 (one time): pull the embedding model in Ollama if using local provider
#   ollama pull nomic-embed-text

# Step 2 (one time per PDF): ingest
# ingest_pdf("path/to/medical_guidelines.pdf")

# Step 3: query
response = medical_rag("What causes high blood pressure?")
response